# Notebook 2 — M/M/c Multi-Server Queue

**Series:** DES Playground · Discrete Event Simulation in SimPy  
**Prerequisites:** Notebook 1 (M/M/1 basics)

**What you'll learn:**
- Extend M/M/1 to c parallel servers using `simpy.Resource(capacity=c)`
- Implement the **Erlang-C formula** and compare it with simulation
- Find the **minimum number of servers** needed to meet a wait-time SLA
- Understand **diminishing returns**: each extra server helps less than the last

---

## 1 · Background — M/M/c Queue

With **c** identical servers the utilisation per server is **ρ = λ/(c·μ)**.  
The steady-state probability that an arriving customer has to wait is given
by the **Erlang-C formula** `C(c, a)` where **a = λ/μ** (offered load):

```
C(c, a) = (a^c / c!) · (c/(c−a)) / [ Σ_{k=0}^{c-1} a^k/k! + (a^c/c!) · c/(c−a) ]

Wq  = C(c,a) / (c·μ − λ)   mean wait in queue
W   = Wq + 1/μ             mean sojourn time
Lq  = λ · Wq               (Little's Law)
L   = λ · W
```

In SimPy, adding more servers is a one-line change:
```python
resource = simpy.Resource(env, capacity=c)  # c servers share one queue
```

In [ ]:
!pip install simpy --quiet

In [ ]:
import simpy
import random
import math
import statistics
from typing import List, Dict
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams.update({'figure.dpi': 110, 'font.size': 11})

# ── Shared helpers (same as Notebook 1) ───────────────────────────────────────
T_TABLE = {1:12.706,2:4.303,3:3.182,4:2.776,5:2.571,6:2.447,7:2.365,
           8:2.306,9:2.262,10:2.228,15:2.131,20:2.086,25:2.060,30:2.042}

def t_crit(df):
    if df in T_TABLE: return T_TABLE[df]
    if df > 30: return 1.960
    keys = sorted(T_TABLE)
    for i in range(len(keys)-1):
        lo, hi = keys[i], keys[i+1]
        if lo < df < hi:
            return T_TABLE[lo] + (T_TABLE[hi]-T_TABLE[lo])*(df-lo)/(hi-lo)
    return 2.0

def ci(values):
    n = len(values)
    if n < 2: return dict(mean=values[0] if values else 0, half=0, lo=0, hi=0)
    mu = statistics.mean(values)
    s  = statistics.stdev(values)
    h  = t_crit(n-1) * s / math.sqrt(n)
    return dict(mean=mu, half=h, lo=mu-h, hi=mu+h)

print('Helpers loaded.')

## 2 · Erlang-C Formula (Theoretical Benchmark)

In [ ]:
def erlang_c(c: int, lam: float, mu: float) -> Dict:
    """
    Compute M/M/c steady-state metrics.
    Returns dict with rho, C (Erlang-C probability), Wq, W, Lq, L.
    Returns None if system is unstable (rho >= 1).
    """
    a   = lam / mu          # offered load (Erlangs)
    rho = lam / (c * mu)    # per-server utilisation

    if rho >= 1.0:
        return None  # unstable

    # Denominator terms
    sum_terms = sum(a**k / math.factorial(k) for k in range(c))
    last_term = (a**c / math.factorial(c)) * (c / (c - a))
    C = last_term / (sum_terms + last_term)  # P(wait > 0)

    Wq = C / (c * mu - lam)   # mean queue wait
    W  = Wq + 1 / mu
    Lq = lam * Wq
    L  = lam * W

    return dict(rho=rho, C=C, Wq=Wq, W=W, Lq=Lq, L=L)

# Quick test: M/M/2, λ=1.6, μ=1.0  → rho=0.8
th = erlang_c(2, 1.6, 1.0)
print('M/M/2 theory (λ=1.6, μ=1.0):')
for k, v in th.items():
    print(f'  {k}: {v:.4f}')

## 3 · M/M/c Simulation

In [ ]:
def run_mmc(
    arrival_rate: float,
    service_rate: float,
    servers:      int,
    horizon:      float = 20_000,
    warmup:       float = 1_000,
    seed:         int   = 42,
) -> Dict:
    rng = random.Random(seed)
    env = simpy.Environment()
    # ONE resource with capacity=servers.  SimPy handles the shared queue.
    pool = simpy.Resource(env, capacity=servers)

    last_t  = [0.0]
    q_area  = [0.0]
    b_area  = [0.0]
    wait_times:    List[float] = []
    sojourn_times: List[float] = []
    departures = [0]

    def integrate(t):
        dt = t - last_t[0]
        q_area[0] += len(pool.queue) * dt
        b_area[0] += pool.count * dt
        last_t[0]  = t

    def entity(arrival_t):
        track = arrival_t >= warmup
        integrate(env.now)
        q_enter = env.now
        with pool.request() as req:
            yield req
            integrate(env.now)
            wait = env.now - q_enter
            svc  = rng.expovariate(service_rate)
            yield env.timeout(svc)
            integrate(env.now)
        departures[0] += 1
        if track:
            wait_times.append(wait)
            sojourn_times.append(wait + svc)

    def source():
        while True:
            yield env.timeout(rng.expovariate(arrival_rate))
            env.process(entity(env.now))

    env.process(source())
    env.run(until=horizon)
    integrate(horizon)

    obs = max(horizon - warmup, 1.0)
    return {
        'lq':  q_area[0] / obs,
        'rho': (b_area[0] / obs) / servers,
        'wq':  statistics.mean(wait_times) if wait_times else 0,
        'w':   statistics.mean(sojourn_times) if sojourn_times else 0,
        'throughput': departures[0] / obs,
    }

print('run_mmc() defined.')

## 4 · Compare 1, 2, 3, 4 Servers at the Same Arrival Rate

We fix **λ = 3.6** and **μ = 1.0** (offered load = 3.6 Erlangs).
With 1 server ρ > 1 (unstable). We increase servers until the queue is comfortable.

In [ ]:
LAMBDA = 3.6
MU     = 1.0

print(f'Arrival rate λ={LAMBDA}, Service rate μ={MU}')
print(f'Offered load a = λ/μ = {LAMBDA/MU:.1f} Erlangs')
print()
print(f'{"c":>4} {"ρ(sim)":>8} {"ρ(theory)":>10} {"Lq(sim)":>9} {"Lq(theory)":>11} {"Wq(sim)":>9} {"Wq(theory)":>11}')
print('-' * 70)

results_by_c = {}

for c in range(1, 7):
    rho_th = LAMBDA / (c * MU)
    if rho_th >= 1.0:
        print(f'{c:>4}  UNSTABLE (ρ={rho_th:.2f} ≥ 1)')
        continue

    th = erlang_c(c, LAMBDA, MU)
    runs = [run_mmc(LAMBDA, MU, c, horizon=20_000, warmup=1_000, seed=42+k*1000) for k in range(5)]
    sim_rho = ci([r['rho'] for r in runs])
    sim_lq  = ci([r['lq']  for r in runs])
    sim_wq  = ci([r['wq']  for r in runs])

    results_by_c[c] = dict(th=th, sim_lq=sim_lq, sim_wq=sim_wq, sim_rho=sim_rho)
    print(f'{c:>4} {sim_rho["mean"]:>8.3f} {th["rho"]:>10.3f} {sim_lq["mean"]:>9.3f} {th["Lq"]:>11.3f} {sim_wq["mean"]:>9.3f} {th["Wq"]:>11.3f}')

In [ ]:
valid_c = sorted(results_by_c.keys())
lq_sims = [results_by_c[c]['sim_lq']['mean'] for c in valid_c]
lq_cihi = [results_by_c[c]['sim_lq']['hi']   for c in valid_c]
lq_cilo = [results_by_c[c]['sim_lq']['lo']   for c in valid_c]
lq_th   = [results_by_c[c]['th']['Lq']       for c in valid_c]

wq_sims = [results_by_c[c]['sim_wq']['mean'] for c in valid_c]
wq_th   = [results_by_c[c]['th']['Wq']       for c in valid_c]

x = np.arange(len(valid_c))
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Lq
axes[0].bar(x - 0.2, lq_sims, 0.35, label='Simulation', color='steelblue', alpha=0.8)
axes[0].bar(x + 0.2, lq_th,   0.35, label='Theory (Erlang-C)', color='coral', alpha=0.8)
axes[0].errorbar(x - 0.2, lq_sims,
                 yerr=[np.array(lq_sims)-np.array(lq_cilo), np.array(lq_cihi)-np.array(lq_sims)],
                 fmt='none', color='black', capsize=4)
axes[0].set_xticks(x)
axes[0].set_xticklabels([f'c={c}' for c in valid_c])
axes[0].set_ylabel('Mean queue length Lq')
axes[0].set_title('Queue length vs number of servers')
axes[0].legend()

# Wq
axes[1].bar(x - 0.2, wq_sims, 0.35, label='Simulation', color='steelblue', alpha=0.8)
axes[1].bar(x + 0.2, wq_th,   0.35, label='Theory (Erlang-C)', color='coral', alpha=0.8)
axes[1].set_xticks(x)
axes[1].set_xticklabels([f'c={c}' for c in valid_c])
axes[1].set_ylabel('Mean wait time Wq')
axes[1].set_title('Wait time vs number of servers')
axes[1].legend()

plt.suptitle(f'M/M/c Comparison — λ={LAMBDA}, μ={MU}', fontsize=13)
plt.tight_layout()
plt.show()

## 5 · Finding the Minimum Server Count for an SLA

In practice you have a **Service Level Agreement** (SLA): e.g.
*"95 % of customers should wait less than 2 minutes"*.

The probability of waiting longer than *t* is:
```
P(wait > t) = C(c, a) · exp(−(c·μ − λ)·t)
```

We find the minimum **c** such that P(wait > SLA_t) ≤ SLA_pct.

In [ ]:
SLA_T   = 2.0   # max acceptable wait time
SLA_PCT = 0.05  # at most 5 % of customers wait longer than SLA_T

print(f'SLA: P(wait > {SLA_T}) ≤ {SLA_PCT*100:.0f}%')
print(f'λ={LAMBDA}, μ={MU}')
print()

for c in range(1, 15):
    th = erlang_c(c, LAMBDA, MU)
    if th is None:
        print(f'c={c:2d}: UNSTABLE')
        continue
    # P(wait > SLA_T)
    p_wait = th['C'] * math.exp(-(c * MU - LAMBDA) * SLA_T)
    meets  = '✓ MEETS SLA' if p_wait <= SLA_PCT else '✗'
    print(f'c={c:2d}: P(wait>{SLA_T}) = {p_wait:.4f}  ρ={th["rho"]:.3f}  Wq={th["Wq"]:.3f}   {meets}')
    if p_wait <= SLA_PCT:
        print(f'\n→ Minimum servers needed: {c}')
        break

## 6 · Utilisation vs Wait — The Efficiency Frontier

More servers → lower wait, but also lower utilisation per server
(= higher cost).  Plot both together to find the sweet spot.

In [ ]:
c_vals  = range(4, 10)
rho_vals_th = []
wq_vals_th  = []

for c in c_vals:
    th = erlang_c(c, LAMBDA, MU)
    if th:
        rho_vals_th.append(th['rho'])
        wq_vals_th.append(th['Wq'])

fig, ax1 = plt.subplots(figsize=(9, 5))
ax2 = ax1.twinx()

ax1.bar(list(c_vals), rho_vals_th, color='steelblue', alpha=0.6, label='Utilisation ρ (left)')
ax2.plot(list(c_vals), wq_vals_th, 'ro-', ms=7, label='Mean wait Wq (right)')

ax1.set_xlabel('Number of servers c')
ax1.set_ylabel('Per-server utilisation ρ', color='steelblue')
ax2.set_ylabel('Mean wait Wq (time units)', color='red')
ax1.set_title(f'Efficiency frontier — λ={LAMBDA}, μ={MU}')
ax2.axhline(SLA_T, color='green', ls='--', label=f'SLA Wq ≤ {SLA_T}')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper right')
plt.tight_layout()
plt.show()

## 7 · Summary

| Insight | Detail |
|---------|--------|
| **SimPy change** | Adding c servers is just `Resource(env, capacity=c)` |
| **Erlang-C** | Closed-form theory — matches simulation within CI |
| **Diminishing returns** | 4→5 servers helps less than 3→4 |
| **Utilisation cost** | More servers → lower utilisation → higher idle cost |
| **SLA analysis** | Erlang-C lets you find minimum c without brute-force simulation |

**Next:** [Notebook 3](03_md1_vs_mm1_variability.ipynb) — Does distribution shape matter?